# EvolveAI — T4 Max-GPU One-Shot SNN Training

One full training invocation. Four approved Hugging Face streams feed the bounded Gatekeeper; mixed precision and a larger recurrent SNN configuration increase useful GPU work. This remains a research prototype, not AGI or a conscious system.

**This notebook does not run benchmark sweeps or repeated test jobs.** Validation evaluations are retained inside the single training run for early stopping and checkpoint selection. Colab GPU quotas and session limits still apply.

In [ ]:
from pathlib import Path
import subprocess, sys
REPO_DIR = Path('/content/EvolveAI')
BRANCH = 'evolveai-maxgpu-one-shot'
if not REPO_DIR.exists():
    subprocess.run(['git', 'clone', '--branch', BRANCH, '--single-branch', 'https://github.com/swasthik-s/EvolveAI.git', str(REPO_DIR)], check=True)
else:
    subprocess.run(['git', '-C', str(REPO_DIR), 'fetch', 'origin', BRANCH], check=True)
    subprocess.run(['git', '-C', str(REPO_DIR), 'checkout', BRANCH], check=True)
    subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only', 'origin', BRANCH], check=True)
%cd /content/EvolveAI/snn
%pip -q install 'datasets>=3.0,<5.0'
print('Repository branch:', BRANCH)


In [ ]:
import torch
if not torch.cuda.is_available():
    raise RuntimeError('CUDA GPU not detected. Select Runtime > Change runtime type > T4 GPU.')
props = torch.cuda.get_device_properties(0)
print('PyTorch:', torch.__version__)
print('GPU:', props.name)
print(f'VRAM: {props.total_memory / 1024**3:.2f} GiB')
print('BF16 supported:', torch.cuda.is_bf16_supported())
print('CUDA:', torch.version.cuda)
print('This run uses automatic BF16/FP16 mixed precision; T4 normally selects FP16.')


## Persistent storage

Mount Drive before training. Checkpoints, report and episodic memory are written there.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
RUN_DIR = Path('/content/drive/MyDrive/EvolveAI/maxgpu_run')
RUN_DIR.mkdir(parents=True, exist_ok=True)
print('Output:', RUN_DIR)


## Single training configuration

T4-oriented starting profile: batch 64, sequence 128, hidden 256, embedding 96. These values are intentionally larger than the previous 8 / 32 / 64 profile. They are not guaranteed to fit every GPU/runtime variant; the run will stop with a clear CUDA OOM if the assigned device has less memory than expected.

In [ ]:
# ONE TRAINING JOB — no sweeps, no separate benchmark runs
import importlib
import train, streaming_gatekeeper, streaming_train, streaming_one_shot
importlib.reload(train)
importlib.reload(streaming_gatekeeper)
importlib.reload(streaming_train)
importlib.reload(streaming_one_shot)
from streaming_one_shot import run_streaming_one_shot

SOURCE_IDS = ['tinystories', 'wikitext', 'ag_news', 'gsm8k']
print('Approved streams:', SOURCE_IDS)
print('Starting the only training invocation now.')

report = run_streaming_one_shot(
    output_dir=RUN_DIR,
    source_ids=SOURCE_IDS,
    steps=8000,
    eval_interval=200,
    patience=15,
    min_delta=0.001,
    block_size=128,
    batch_size=64,
    hidden_size=256,
    embedding_dim=96,
    learning_rate=0.001,
    seed=42,
    architecture='baseline',
    device='cuda',
)

print('\n=== FINAL TRAINING REPORT ===')
print('Steps:', report['steps_completed'])
print('Best step:', report['best_step'])
print('Best validation loss:', report['best_validation_loss'])
print('Stopped early:', report['stopped_early'])
print('Elapsed seconds:', report['elapsed_seconds'])
print('Report:', RUN_DIR / 'streaming_one_shot_report.json')
print('Best checkpoint:', report['checkpoint_path'])
print('Latest recovery checkpoint:', RUN_DIR / 'streaming_one_shot_latest.pt')
print('Episodic memory:', RUN_DIR / 'episodic_memory.jsonl')


In [ ]:
import json
report = json.loads((RUN_DIR / 'streaming_one_shot_report.json').read_text(encoding='utf-8'))
print('Configuration:', report['training_config'])
print('Gatekeeper admitted:', report['gatekeeper']['records_admitted'])
print('Gatekeeper rejected:', report['gatekeeper']['records_rejected'])
print('Source statistics:')
for source, counts in report['gatekeeper']['sources'].items():
    print(' ', source, counts)
print('Memory events:', report['episodic_memory']['events'])
print('Learning director decisions:', report['learning_director']['decisions'])


## Interpretation and limitations

- The Gatekeeper streams records; it does not download entire datasets.
- The model predicts the next byte. It is not yet a structured reasoning or conversational model.
- The online validation buffer is useful for monitoring but is not an independent benchmark.
- A checkpoint is a trained model state, not a complete resumable training session with optimizer and stream cursor state.
- GPU utilization depends on Colab's assigned accelerator, streaming/network throughput and the Python-unrolled recurrent SNN. This configuration increases GPU work but cannot guarantee 100% utilization.
- Review dataset cards, licenses and source-content rights before use beyond research.